# GCTA-COJO conditional analysis for PD age at onset (AAO)

Identifies additional independent signals in the full joint AAO meta-analysis (EUR) using GCTA-COJO (`--cojo-slct`).

**Workflow**
1. Build a per-chromosome LD reference panel from GP2 EUR imputed genotypes, restricted to PD cases with AAO/AAD
2. Format the meta-analysis summary statistics for COJO
3. Run COJO per chromosome

Jobs run on Google Cloud Batch, with GCS buckets mounted via gcsfuse.

## Setup

In [ ]:
# Requires google-cloud-batch (pip install google-cloud-batch)
import pandas as pd
from google.cloud import batch_v1

### Configuration

In [ ]:
# Google Cloud
PROJECT_ID = "<gcp-project-id>"
REGION = "<gcp-region>"

# Buckets and local mount points
RELEASE_BUCKET = "<gp2-release-bucket>"   # GP2 release (imputed genotypes)
WORKING_BUCKET = "<working-bucket>"       # inputs/outputs for this analysis
RELEASE_MOUNT = "gp2_release"
WORKING_MOUNT = "gp2_working"

# GP2 release
RELEASE = "release11"

# Analysis directories (within the working bucket)
ANALYSIS_DIR = "GP2_AAO"
COJO_DIR = f"{ANALYSIS_DIR}/COJO"

# Inputs
CASES_KEEP = f"{ANALYSIS_DIR}/<AAO_cases_keep_file>.txt"   # PD cases with AAO/AAD used in the GWAS (FID IID)
SUMSTATS = "<path/to/AAO_meta_analysis_sumstats>.txt.gz"   # full joint EUR AAO meta-analysis (hg38)

Mount the release and working buckets.

In [ ]:
! mkdir -p ~/{RELEASE_MOUNT} ~/{WORKING_MOUNT}
! gcsfuse --dir-mode 777 --file-mode 777 --implicit-dirs {WORKING_BUCKET} ~/{WORKING_MOUNT}
! gcsfuse --dir-mode 777 --file-mode 777 --implicit-dirs {RELEASE_BUCKET} ~/{RELEASE_MOUNT}

### Cloud Batch job helper
Submits a single-task job that runs `script_text` on an `e2-standard-8` VM (8 CPUs, 32 GB) with the input and output buckets mounted.

In [ ]:
def create_script_job_with_buckets(
    project_id: str, region: str, job_name: str, bucket_name_input: str, bucket_name_output: str, script_text: str) -> batch_v1.Job:
    """Create a Cloud Batch job that runs script_text with input/output buckets mounted.
    job_name must be unique per project and region."""
    client = batch_v1.BatchServiceClient()

    task = batch_v1.TaskSpec()
    runnable = batch_v1.Runnable()

    runnable.script = batch_v1.Runnable.Script()
    runnable.script.text = script_text
    task.runnables = [runnable]

    # Input bucket -> ./RELEASE_MOUNT
    gcs_bucket_input = batch_v1.GCS(remote_path=bucket_name_input)
    gcs_volume_input = batch_v1.Volume(gcs=gcs_bucket_input, mount_path=f"./{RELEASE_MOUNT}")

    # Output bucket -> ./WORKING_MOUNT
    gcs_bucket_output = batch_v1.GCS(remote_path=bucket_name_output)
    gcs_volume_output = batch_v1.Volume(gcs=gcs_bucket_output, mount_path=f"./{WORKING_MOUNT}")

    task.volumes = [gcs_volume_input, gcs_volume_output]

    # Compute resources per task
    resources = batch_v1.ComputeResource()
    resources.cpu_milli = 8000  # 8 CPUs
    resources.memory_mib = 32768 # 32 GB
    task.compute_resource = resources

    task.max_retry_count = 1
    task.max_run_duration = "86400s"

    group = batch_v1.TaskGroup()
    group.task_count = 1
    group.task_spec = task

    # VM type
    allocation_policy = batch_v1.AllocationPolicy()
    policy = batch_v1.AllocationPolicy.InstancePolicy()
    policy.machine_type = "e2-standard-8"
    instances = batch_v1.AllocationPolicy.InstancePolicyOrTemplate()
    instances.policy = policy
    allocation_policy.instances = [instances]

    job = batch_v1.Job()
    job.task_groups = [group]
    job.allocation_policy = allocation_policy
    job.labels = {"env": "testing", "type": "script", "mount": "bucket"}
    job.logs_policy = batch_v1.LogsPolicy()
    job.logs_policy.destination = batch_v1.LogsPolicy.Destination.CLOUD_LOGGING

    create_request = batch_v1.CreateJobRequest()
    create_request.job = job
    create_request.job_id = job_name
    create_request.parent = f"projects/{project_id}/locations/{region}"

    return client.create_job(create_request)

### Tool install scripts
Run at the start of each Batch job to install PLINK2 and GCTA into `$HOME/bin` on the VM.

In [ ]:
%%writefile ./{WORKING_MOUNT}/{ANALYSIS_DIR}/plink2_module.sh
#!/bin/bash

# Set the PLINK version
PLINK_VERSION="2.3"

sudo apt update
sudo apt install -y unzip

# PLINK2 download URL
PLINK_URL="https://s3.amazonaws.com/plink2-assets/alpha7/plink2_linux_x86_64_20260504.zip"

# Set the VM directory where PLINK will be installed
PLINK_INSTALL_DIR="$HOME/bin/plink2"

# Create the PLINK installation directory
mkdir -p $PLINK_INSTALL_DIR

# Download and unzip the latest PLINK release
curl -L $PLINK_URL -o plink2.zip
unzip plink2.zip -d $PLINK_INSTALL_DIR

# Create the module file
mkdir -p /etc/modulefiles/plink2
cat <<EOF > /etc/modulefiles/plink2/${PLINK_VERSION}
#%Module
set plink_root ${PLINK_INSTALL_DIR}/plink2
prepend-path PATH $plink_root
prepend-path LD_LIBRARY_PATH $plink_root
EOF

In [ ]:
%%writefile ./{WORKING_MOUNT}/{ANALYSIS_DIR}/gcta_module.sh
#!/bin/bash

sudo apt update
sudo apt install -y unzip

# Set the URL for the current GCTA release
GCTA_VERSION="1.95.3"
GCTA_DIRNAME="gcta-${GCTA_VERSION}-linux-x86_64"
GCTA_URL="https://yanglab.westlake.edu.cn/software/gcta/bin/${GCTA_DIRNAME}.zip"

# Set the VM directory where GCTA will be installed
GCTA_INSTALL_DIR="$HOME/bin/gcta"

mkdir -p $GCTA_INSTALL_DIR

# Download and unzip, failing loudly if the download isn't a real file
curl -fL $GCTA_URL -o ${GCTA_DIRNAME}.zip
file ${GCTA_DIRNAME}.zip
unzip ${GCTA_DIRNAME}.zip -d $GCTA_INSTALL_DIR

# Create the module file (modulefiles/<n>/<version>, not modulefiles/<n>)
mkdir -p /etc/modulefiles/gcta
cat <<EOF > /etc/modulefiles/gcta/${GCTA_VERSION}
#%Module
set gcta_root ${GCTA_INSTALL_DIR}/${GCTA_DIRNAME}
prepend-path PATH \$gcta_root
prepend-path LD_LIBRARY_PATH \$gcta_root
EOF

## Step 1: Build LD reference panel
Subset GP2 EUR imputed genotypes to PD cases with AAO/AAD (the individuals in the AAO GWAS) and convert to PLINK1 binary format, one job per chromosome.

In [ ]:
# Write per-chromosome subsetting script
release_path = f'./{RELEASE_MOUNT}/{RELEASE}/imputed_genotypes/EUR/chr${{CHROM}}_EUR_{RELEASE}_vwb'
out_dir = f'./{WORKING_MOUNT}/{COJO_DIR}/chr${{CHROM}}.EUR_for_COJO'

subset_cmd = f"$HOME/bin/plink2/plink2 --pfile {release_path} \
     --maf 0.0001 \
     --keep ./{WORKING_MOUNT}/{CASES_KEEP} \
     --make-bed \
     --out {out_dir}"

with open(f'./{WORKING_MOUNT}/{ANALYSIS_DIR}/make_cojo_chrs.sh', 'w') as f:
    f.write('#!/bin/bash\n\n')
    f.write('CHROM=$1\n\n')
    f.write(f'{subset_cmd}\n')
    f.close()

In [ ]:
# Submit one job per chromosome (job names must be unique; change the prefix to rerun)
for chrom in range(1, 23):
    job_name = f"aao-cojo-ref-{chrom}"

    script = f"""
        #!/bin/bash

        # Install and activate PLINK2
        chmod +x  ./{WORKING_MOUNT}/{ANALYSIS_DIR}/plink2_module.sh
        ./{WORKING_MOUNT}/{ANALYSIS_DIR}/plink2_module.sh
    
        # Run subsetting for this chromosome
        chmod +x ./{WORKING_MOUNT}/{ANALYSIS_DIR}/make_cojo_chrs.sh
        ./{WORKING_MOUNT}/{ANALYSIS_DIR}/make_cojo_chrs.sh {chrom}
        wait
        """

    create_script_job_with_buckets(project_id = PROJECT_ID, region = REGION, 
                    job_name = job_name, bucket_name_input= RELEASE_BUCKET, bucket_name_output= WORKING_BUCKET, 
                    script_text = script)

## Step 2: Format summary statistics for COJO
Full joint AAO meta-analysis (EUR, hg38). COJO format: `SNP A1 A2 freq b se p N`, with `N` = number of cases.

In [ ]:
stats = pd.read_csv(SUMSTATS, compression='gzip', sep='\t')

stats_cojo = stats[['variant_ID', 'alt_allele', 'ref_allele', 'effect_allele_frequency', 'beta', 'standard_error', 'p_value', 'n_cases']]
stats_cojo.columns = ['SNP', 'A1', 'A2', 'freq', 'b', 'se', 'p', 'N']
stats_cojo.to_csv(f'./{WORKING_MOUNT}/{COJO_DIR}/AAO_stats_for_cojo.txt', index=False, sep='\t')

## Step 3: Run COJO
Stepwise selection of independent signals (`--cojo-slct`) per chromosome against the Step 1 reference panel. Outputs `chr*.AAO_COJO.*` in the COJO directory.

In [ ]:
# Write per-chromosome COJO script
GCTA_VERSION = "1.95.3"
GCTA_DIRNAME = f"gcta-{GCTA_VERSION}-linux-x86_64"

release_path = f'./{WORKING_MOUNT}/{COJO_DIR}/chr${{CHROM}}.EUR_for_COJO'
out_dir = f'./{WORKING_MOUNT}/{COJO_DIR}/chr${{CHROM}}.AAO_COJO'

subset_cmd = f"$HOME/bin/gcta/{GCTA_DIRNAME}/gcta --bfile {release_path} \
     --chr ${{CHROM}} --maf 0.001 \
     --cojo-file ./{WORKING_MOUNT}/{COJO_DIR}/AAO_stats_for_cojo.txt \
     --cojo-slct \
     --out {out_dir}"

with open(f'./{WORKING_MOUNT}/{COJO_DIR}/aao_cojo.sh', 'w') as f:
    f.write('#!/bin/bash\n\n')
    f.write('CHROM=$1\n\n')
    f.write(f'{subset_cmd}\n')
    f.close()

In [ ]:
# Submit one job per chromosome (job names must be unique; change the prefix to rerun)
for chrom in range(1, 23):
    job_name = f"aao-cojo-run-{chrom}"

    script = f"""
        #!/bin/bash

        # Install and activate gcta
        chmod +x  ./{WORKING_MOUNT}/{ANALYSIS_DIR}/gcta_module.sh
        ./{WORKING_MOUNT}/{ANALYSIS_DIR}/gcta_module.sh
    
        # Run COJO for this chromosome
        chmod +x ./{WORKING_MOUNT}/{COJO_DIR}/aao_cojo.sh
        ./{WORKING_MOUNT}/{COJO_DIR}/aao_cojo.sh {chrom}
        wait
        """

    create_script_job_with_buckets(project_id = PROJECT_ID, region = REGION, 
                    job_name = job_name, bucket_name_input= WORKING_BUCKET, bucket_name_output= WORKING_BUCKET, 
                    script_text = script)